Paper: https://arxiv.org/pdf/1602.07360

# What is the goal?

-  identify a model that has very few parameters while preserving accuracy

# What are the architectural innovations?

- replace 3x3 with 1x1 convs
- decrease number of channels to 3x3 filters
- Downsample late in the network so that convolution layers have large activation maps.

# What are Fire Modules?

- contains a squeeze and expand layer, taking in number of squeeze filters (1x1), expand filters (1x1), and expand filters (3x3)

- squeeze layer only contains 1x1 filters

- excite layers contains two branches (1x1 conv) and (3x3 conv) whose results are then concatenated.

- 1-pixel border of zero-padding to the input data of the 3x3 filters to maintain spatial dimensions.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch import nn
from torchvision import transforms, datasets
from torch.utils.data import Subset

class FireModule(nn.Module):
    def __init__(self, in_channels, sqz_out_channels, expand_filters_one, expand_filters_three):
        super().__init__()

        self.squeeze_layers = nn.Conv2d(in_channels, sqz_out_channels, kernel_size=1)

        self.expand_ones = nn.Sequential(
            nn.Conv2d(sqz_out_channels, expand_filters_one, kernel_size=1),
            nn.ReLU()            
        )

        self.expand_threes = nn.Sequential(
            nn.Conv2d(sqz_out_channels, expand_filters_three, kernel_size=3, padding=1),
            nn.ReLU()            
        )

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.squeeze_layers(x)

        return torch.cat([self.expand_ones(x), self.expand_threes(x)], dim=1)


class SqueezeNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.Sequential(
            # nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=2),
            nn.Conv2d(in_channels=3, out_channels=96, kernel_size=3, stride=1, padding=1),

            nn.ReLU(), # not specified clearly within paper
            # nn.MaxPool2d(kernel_size=3, stride=2),
            FireModule(96, sqz_out_channels=16, expand_filters_one=64, expand_filters_three=64),
            FireModule(128, sqz_out_channels=16, expand_filters_one=64, expand_filters_three=64),
            FireModule(128, sqz_out_channels=32, expand_filters_one=128, expand_filters_three=128),

            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            FireModule(256, sqz_out_channels=32, expand_filters_one=128, expand_filters_three=128),
            FireModule(256, sqz_out_channels=48, expand_filters_one=192, expand_filters_three=192),
            FireModule(384, sqz_out_channels=48, expand_filters_one=192, expand_filters_three=192),
            FireModule(384, sqz_out_channels=64, expand_filters_one=256, expand_filters_three=256),

            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            FireModule(512, sqz_out_channels=64, expand_filters_one=256, expand_filters_three=256),
            # nn.Dropout(0.5),
            nn.Conv2d(in_channels=512, out_channels=100, kernel_size=1, stride=1),
            nn.AdaptiveAvgPool2d((1, 1))
        )
    
    def forward(self, x):
        x = self.layers(x)
        x = x.view(x.size(0), -1)
        return x

In [16]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Mild to avoid over-distortion
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4865, 0.4409], std=[0.2673, 0.2564, 0.2761]),
    transforms.RandomErasing(p=0.25)  # Apply after normalization for consistency
])

test_transform = transforms.Compose([
    transforms.ToTensor(), # Moved ToTensor before Normalize (good practice)
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

# Load raw datasets
cifar_train_raw = datasets.CIFAR100(root="./data", train=True, download=True, transform=None)

train_size = int(0.9 * len(cifar_train_raw))  # 48,000

train_indices = list(range(0, train_size))
val_indices = list(range(train_size, len(cifar_train_raw)))

# Create datasets with appropriate transforms
cifar_train = Subset(
    datasets.CIFAR100(root="./data", train=True, transform=train_transform),
    train_indices
)
cifar_val = Subset(
    datasets.CIFAR100(root="./data", train=True, transform=test_transform),
    val_indices
)
# Use original test set (10,000 samples) - close to 10% of 60,000
cifar_test = datasets.CIFAR100(root="./data", train=False, transform=test_transform)

train_loader = DataLoader(
    cifar_train,
    batch_size=512,  # Changed from 512
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=6
)

val_loader = DataLoader(
    cifar_val,  # Use directly
    batch_size=512,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=6
)

test_loader = DataLoader(
    cifar_test,
    batch_size=512,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=6
)

num_classes = 100

model = SqueezeNet().to(device)

num_epochs = 40
loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,  # Changed from 1e-3
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,  # Changed from 3e-3
    epochs=num_epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25.0,
    final_div_factor=1000.0
)

best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f'Starting Epoch {epoch+1}')
    model.train()

    current_loss = 0.0
    num_batches = 0

    for i, data in enumerate(train_loader):
        inputs, targets = data
        inputs, targets = inputs.to(device), targets.to(device)
            

        optimizer.zero_grad(set_to_none=True)

        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Changed from 1.0
        
        optimizer.step()
        scheduler.step()

        current_loss += loss.item()
        num_batches += 1

        if i % 50 == 0:
            print(f'Batch {i}/{len(train_loader)}, Loss: {loss.item():.4f}')


    avg_train_loss = current_loss / num_batches
    print(f'Epoch {epoch+1} finished')
    print(f'Training - Loss: {avg_train_loss:.4f}')

    if (epoch + 1) % 2 == 0:
        model.eval()
        val_loss = 0.0
        val_batches = 0

        print(f'Epoch {epoch+1} finished')
        print(f'average training loss is {avg_train_loss:.4f}')

        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_targets = val_data
                val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)  # Convert inputs to FP16

                val_outputs = model(val_inputs)
                val_batch_loss = loss_function(val_outputs, val_targets)

                val_loss += val_batch_loss.item()
                val_batches += 1


        avg_val_loss = val_loss / val_batches

        print(f'Epoch {epoch+1} finished')
        print(f'Training - Loss: {avg_train_loss:.4f}')
        print(f'Validation - Loss: {avg_val_loss:.4f}')

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Using device: cuda
GPU Memory: 15.8 GB
Starting Epoch 1
Batch 0/88, Loss: 4.6048
Batch 50/88, Loss: 4.6060
Epoch 1 finished
Training - Loss: 4.6060
Starting Epoch 2
Batch 0/88, Loss: 4.6050
Batch 50/88, Loss: 4.5891


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b4631fddf80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b4631fddf80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch 2 finished
Training - Loss: 4.5835
Epoch 2 finished
average training loss is 4.5835
Epoch 2 finished
Training - Loss: 4.5835
Validation - Loss: 4.5403
Starting Epoch 3
Batch 0/88, Loss: 4.5365
Batch 50/88, Loss: 4.5424
Epoch 3 finished
Training - Loss: 4.5266
Starting Epoch 4
Batch 0/88, Loss: 4.4845
Batch 50/88, Loss: 4.3841
Epoch 4 finished
Training - Loss: 4.3751
Epoch 4 finished
average training loss is 4.3751
Epoch 4 finished
Training - Loss: 4.3751
Validation - Loss: 4.2793
Starting Epoch 5
Batch 0/88, Loss: 4.3033
Batch 50/88, Loss: 4.2535
Epoch 5 finished
Training - Loss: 4.2683
Starting Epoch 6
Batch 0/88, Loss: 4.1747
Batch 50/88, Loss: 4.1339
Epoch 6 finished
Training - Loss: 4.1483
Epoch 6 finished
average training loss is 4.1483
Epoch 6 finished
Training - Loss: 4.1483
Validation - Loss: 4.0648
Starting Epoch 7
Batch 0/88, Loss: 4.0604
Batch 50/88, Loss: 4.0335
Epoch 7 finished
Training - Loss: 4.0412
Starting Epoch 8
Batch 0/88, Loss: 4.0312
Batch 50/88, Loss: 3.883

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b4631fddf80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
         ^  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b4631fddf80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch 32 finished
Training - Loss: 2.0887
Validation - Loss: 2.2971
Starting Epoch 33
Batch 0/88, Loss: 2.0722
Batch 50/88, Loss: 2.0115
Epoch 33 finished
Training - Loss: 2.0584
Starting Epoch 34
Batch 0/88, Loss: 2.0170
Batch 50/88, Loss: 2.1590
Epoch 34 finished
Training - Loss: 2.0295
Epoch 34 finished
average training loss is 2.0295
Epoch 34 finished
Training - Loss: 2.0295
Validation - Loss: 2.2321
Starting Epoch 35
Batch 0/88, Loss: 2.0937
Batch 50/88, Loss: 1.9982
Epoch 35 finished
Training - Loss: 2.0005
Starting Epoch 36
Batch 0/88, Loss: 1.9827
Batch 50/88, Loss: 1.9954
Epoch 36 finished
Training - Loss: 1.9773
Epoch 36 finished
average training loss is 1.9773
Epoch 36 finished
Training - Loss: 1.9773
Validation - Loss: 2.2298
Starting Epoch 37
Batch 0/88, Loss: 1.9211
Batch 50/88, Loss: 2.0604
Epoch 37 finished
Training - Loss: 1.9672
Starting Epoch 38
Batch 0/88, Loss: 1.9732
Batch 50/88, Loss: 1.9962
Epoch 38 finished
Training - Loss: 1.9522
Epoch 38 finished
average trai

In [17]:
def evaluate_test_set(model):
    model.eval()
    correct = 0
    total = 0
    
    print("Starting evaluation...")
    
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            
            # Use autocast for consistency if you trained with it
            outputs = model(images)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the network on the test images: {accuracy:.2f}%')
    return accuracy

print("\n=== Running standard evaluation ===")
standard_accuracy = evaluate_test_set(model)   
print(f'Standard Test Accuracy: {standard_accuracy:.4f} ({standard_accuracy:.2f}%)')


=== Running standard evaluation ===
Starting evaluation...
Accuracy of the network on the test images: 60.31%
Standard Test Accuracy: 60.3100 (60.31%)
